# Friend Message LoRA SFT on Kaggle

This notebook fine-tunes a small causal language model with LoRA supervised fine-tuning on a Telegram `result.json` export.

The task is: given recent chat context, generate the selected friend's next message.

Expected outputs in `/kaggle/working/`:

- A trained LoRA adapter
- Tokenizer files needed for inference
- A small JSON file with synthetic-context generation examples
- Optional interactive chat helpers

## Privacy Warning

Telegram exports can contain highly private information. Do not publish your `result.json`, generated training examples, trained adapter, logs, screenshots, Kaggle notebook output, or generated replies unless every participant has consented and you have reviewed the content carefully.

This notebook avoids printing raw Telegram messages by default. It prints aggregate statistics and a redacted example structure only.


## Environment Setup

This notebook targets Kaggle with a single GPU, commonly an NVIDIA Tesla P100 with 16 GB VRAM. It uses Kaggle input and working directories only.

Kaggle often has most packages installed already. Run this cell if imports fail or if you want current versions of the required libraries.


In [ ]:
%pip install -q -U transformers datasets peft accelerate bitsandbytes gradio sentencepiece


In [ ]:
import torch

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Enable a Kaggle GPU accelerator before running training."
    )


## Configuration

Attach your Telegram export as a Kaggle Dataset, then update `DATA_PATH` if needed.

Typical flow:

1. Create a private Kaggle Dataset containing only your Telegram `result.json`.
2. Attach that Dataset to this notebook.
3. Run the file-listing cell below.
4. Set `DATA_PATH` to the printed path ending in `result.json`.
5. Choose `MODEL_PRESET`.

The default preset is a small Qwen model. If the P100 runs out of memory, restart the notebook session and switch to `smollm2_360m_instruct` or reduce `MAX_CONTEXT_MESSAGES` / `MAX_ANSWER_TOKENS`.


In [ ]:
import gc
import inspect
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORKING_DIR = Path("/kaggle/working")

DATA_PATH = "/kaggle/input/telegram-chat-export/result.json"
OUTPUT_DIR = "/kaggle/working/friend_lora_output"
GENERATION_EXAMPLES_PATH = "/kaggle/working/generation_examples.json"

MODEL_PRESET = "qwen3_0_6b"

MODEL_CONFIGS = {
    "qwen3_0_6b": {
        "model_name": "Qwen/Qwen3-0.6B",
        "max_seq_length": 512,
        "max_answer_tokens": 96,
    },
    "qwen2_5_0_5b_instruct": {
        "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
        "max_seq_length": 768,
        "max_answer_tokens": 96,
    },
    "smollm2_360m_instruct": {
        "model_name": "HuggingFaceTB/SmolLM2-360M-Instruct",
        "max_seq_length": 768,
        "max_answer_tokens": 96,
    },
    "tinyllama_1_1b_chat": {
        "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        "max_seq_length": 512,
        "max_answer_tokens": 96,
    },
}

if MODEL_PRESET not in MODEL_CONFIGS:
    raise ValueError(f"Unknown MODEL_PRESET: {MODEL_PRESET}. Choose one of {list(MODEL_CONFIGS)}")

MODEL_CONFIG = MODEL_CONFIGS[MODEL_PRESET]
MODEL_NAME = MODEL_CONFIG["model_name"]
MAX_SEQ_LENGTH = MODEL_CONFIG["max_seq_length"]
MAX_ANSWER_TOKENS = MODEL_CONFIG["max_answer_tokens"]

TARGET_SPEAKER_INDEX = None
MAX_CONTEXT_MESSAGES = 6
TRAIN_TEST_SPLIT = 0.0
SEED = 42

LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_TRAIN_EPOCHS = 2
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LR_SCHEDULER_TYPE = "cosine"
LOGGING_STEPS = 10
EVAL_STEPS = 100
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 2
MAX_GRAD_NORM = 0.3

random.seed(SEED)
print(f"Model preset: {MODEL_PRESET}")
print(f"Base model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")


In [ ]:
def list_kaggle_input_files(max_files=80):
    if not KAGGLE_INPUT_DIR.exists():
        print("/kaggle/input does not exist in this environment.")
        return []

    files = sorted(path for path in KAGGLE_INPUT_DIR.rglob("*") if path.is_file())
    print(f"Files under /kaggle/input: {len(files)}")
    for path in files[:max_files]:
        print(path)
    if len(files) > max_files:
        print(f"... {len(files) - max_files} more files not shown")
    return files


def resolve_data_path(configured_path):
    path = Path(configured_path)
    if path.exists():
        return str(path)

    candidates = sorted(KAGGLE_INPUT_DIR.rglob("result.json")) if KAGGLE_INPUT_DIR.exists() else []
    if len(candidates) == 1:
        print(f"DATA_PATH not found. Auto-detected: {candidates[0]}")
        return str(candidates[0])

    if candidates:
        print("Found multiple result.json files:")
        for candidate in candidates:
            print(candidate)

    raise FileNotFoundError(
        "Could not find DATA_PATH. Attach your Telegram export as a Kaggle Dataset "
        "and set DATA_PATH to the attached result.json path."
    )


list_kaggle_input_files()
DATA_PATH = resolve_data_path(DATA_PATH)
print(f"Using DATA_PATH: {DATA_PATH}")
print(f"Saving adapter to: {OUTPUT_DIR}")


## Load Telegram Export

Telegram `message["text"]` may be a string, a list of strings, or a list of text entity dictionaries. This parser converts those forms to plain text, ignores service records, skips empty text, and sorts messages chronologically.


In [ ]:
SPECIAL_MARKERS = ("<|system|>", "<|context|>", "<|answer|>")


def telegram_text_to_plain(value):
    if isinstance(value, str):
        return value
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(telegram_text_to_plain(item.get("text", "")))
        return "".join(parts)
    return ""


def clean_message_text(text):
    text = str(text).replace("\u00a0", " ")
    for marker in SPECIAL_MARKERS:
        text = text.replace(marker, "")
    return re.sub(r"\s+", " ", text).strip()


def normalize_sender_name(value):
    value = "Unknown" if value is None else str(value).strip()
    return value or "Unknown"


def make_speaker_key(sender_name, sender_id):
    if sender_id is not None and str(sender_id).strip():
        return f"id:{sender_id}"
    return f"name:{sender_name}"


def parse_telegram_export(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    raw_messages = data.get("messages", []) if isinstance(data, dict) else data
    if not isinstance(raw_messages, list):
        raw_messages = []

    parsed = []
    for original_index, record in enumerate(raw_messages):
        if not isinstance(record, dict) or record.get("type") != "message":
            continue

        text = clean_message_text(telegram_text_to_plain(record.get("text", "")))
        if not text:
            continue

        sender_name = normalize_sender_name(
            record.get("from") or record.get("actor") or record.get("author")
        )
        sender_id = record.get("from_id") or record.get("actor_id") or record.get("author_id")
        speaker_key = make_speaker_key(sender_name, sender_id)

        try:
            timestamp = int(record.get("date_unixtime") or 0)
        except (TypeError, ValueError):
            timestamp = 0

        try:
            message_id = int(record.get("id", original_index))
        except (TypeError, ValueError):
            message_id = original_index

        parsed.append(
            {
                "text": text,
                "sender_name": sender_name,
                "sender_id": sender_id,
                "speaker_key": speaker_key,
                "date": record.get("date", ""),
                "timestamp": timestamp,
                "message_id": message_id,
                "original_index": original_index,
            }
        )

    parsed.sort(key=lambda item: (item["timestamp"], item["message_id"], item["original_index"]))
    return parsed


messages = parse_telegram_export(DATA_PATH)
if not messages:
    raise ValueError("No usable text messages were found in the Telegram export.")

speaker_count = len({message["speaker_key"] for message in messages})
first_date = next((message["date"] for message in messages if message.get("date")), "unknown")
last_date = next((message["date"] for message in reversed(messages) if message.get("date")), "unknown")

print(f"Parsed usable text messages: {len(messages):,}")
print(f"Detected speakers: {speaker_count:,}")
print(f"Date range: {first_date} to {last_date}")


## Select Speaker

Choose which speaker should be modeled as the friend. Only aggregate speaker statistics are shown.

Set `TARGET_SPEAKER_INDEX` in the configuration cell to avoid the prompt when rerunning the notebook.


In [ ]:
def build_speaker_stats(parsed_messages):
    counts = Counter(message["speaker_key"] for message in parsed_messages)
    names = {}
    ids = {}

    for message in parsed_messages:
        key = message["speaker_key"]
        names.setdefault(key, message["sender_name"])
        ids.setdefault(key, message["sender_id"])

    total = sum(counts.values())
    rows = []
    for index, (speaker_key, count) in enumerate(counts.most_common()):
        rows.append(
            {
                "index": index,
                "speaker_key": speaker_key,
                "sender_name": names.get(speaker_key, "Unknown"),
                "sender_id": ids.get(speaker_key),
                "message_count": count,
                "percentage": 100.0 * count / total if total else 0.0,
            }
        )
    return rows


speaker_stats = build_speaker_stats(messages)
print(f"{'index':>5}  {'messages':>10}  {'percent':>8}  {'sender_id':<24}  sender_name")
for row in speaker_stats:
    sender_id = str(row["sender_id"] or "")[:24]
    print(
        f"{row['index']:>5}  {row['message_count']:>10,}  {row['percentage']:>7.2f}%  "
        f"{sender_id:<24}  {row['sender_name']}"
    )

selected_index = TARGET_SPEAKER_INDEX
if selected_index is None:
    selected_index = int(input("Enter the speaker index to model as the friend: ").strip())

if selected_index < 0 or selected_index >= len(speaker_stats):
    raise ValueError("Selected speaker index is out of range.")

selected_speaker = speaker_stats[selected_index]
TARGET_SPEAKER_KEY = selected_speaker["speaker_key"]
TARGET_SPEAKER_NAME = selected_speaker["sender_name"]
TARGET_SPEAKER_ID = selected_speaker["sender_id"]

print(f"Selected target speaker index: {selected_index}")
print(f"Selected target speaker name: {TARGET_SPEAKER_NAME}")
print(f"Selected target speaker id: {TARGET_SPEAKER_ID or 'not available'}")


## Build Training Examples

For each message written by the selected friend, the notebook uses the previous `MAX_CONTEXT_MESSAGES` messages as context and trains the model to produce only the friend's next message.

The prompt and context are masked during training, so loss is computed only on the answer tokens.


In [ ]:
SYSTEM_PROMPT = """<|system|>
You are imitating the writing style of a specific person in a private Telegram chat.
Given the recent chat context, write only this person's next message.
Do not explain yourself.
Do not add metadata."""


def role_for_message(message):
    return "Friend" if message["speaker_key"] == TARGET_SPEAKER_KEY else "User"


def format_training_prompt(context_messages):
    context_lines = [f"{role_for_message(message)}: {message['text']}" for message in context_messages]
    return f"{SYSTEM_PROMPT}\n<|context|>\n" + "\n".join(context_lines) + "\n<|answer|>\n"


def build_training_rows(parsed_messages, max_context_messages):
    rows = []
    shapes = []

    for index, message in enumerate(parsed_messages):
        if message["speaker_key"] != TARGET_SPEAKER_KEY:
            continue

        context_messages = parsed_messages[max(0, index - max_context_messages):index]
        if not context_messages:
            continue

        prompt = format_training_prompt(context_messages)
        answer = message["text"]
        rows.append({"prompt": prompt, "answer": answer})
        shapes.append(
            {
                "context_roles": [role_for_message(context_message) for context_message in context_messages],
                "context_message_count": len(context_messages),
                "answer_chars": len(answer),
                "example_chars": len(prompt) + len(answer),
            }
        )

    return rows, shapes


training_rows, example_shapes = build_training_rows(messages, MAX_CONTEXT_MESSAGES)
if not training_rows:
    raise ValueError("No training examples were created. Choose a speaker with messages that have previous context.")

print(f"Created supervised examples: {len(training_rows):,}")


## Dataset Preparation

This section creates Hugging Face datasets and prints safe aggregate statistics only.


In [ ]:
from datasets import Dataset


def percentile(values, pct):
    if not values:
        return 0
    values = sorted(values)
    index = min(len(values) - 1, max(0, round((pct / 100) * (len(values) - 1))))
    return values[index]


def make_redacted_example(shape):
    lines = [f"{role}: [message hidden]" for role in shape["context_roles"]]
    return f"{SYSTEM_PROMPT}\n<|context|>\n" + "\n".join(lines) + "\n<|answer|>\n[message hidden]"


dataset = Dataset.from_list(training_rows).shuffle(seed=SEED)
if len(dataset) > 1 and TRAIN_TEST_SPLIT > 0:
    validation_size = max(1, int(round(len(dataset) * TRAIN_TEST_SPLIT)))
    validation_size = min(validation_size, len(dataset) - 1)
    split_dataset = dataset.train_test_split(test_size=validation_size, seed=SEED)
    train_dataset = split_dataset["train"]
    eval_dataset = split_dataset["test"]
else:
    train_dataset = dataset
    eval_dataset = None

message_counts_by_speaker = Counter(message["speaker_key"] for message in messages)
example_char_lengths = [shape["example_chars"] for shape in example_shapes]
answer_char_lengths = [shape["answer_chars"] for shape in example_shapes]

print(f"Parsed messages: {len(messages):,}")
print(f"Speakers: {len(message_counts_by_speaker):,}")
print(f"Selected target speaker messages: {message_counts_by_speaker[TARGET_SPEAKER_KEY]:,}")
print(f"Training examples: {len(dataset):,}")
print(f"Train size: {len(train_dataset):,}")
print(f"Validation size: {len(eval_dataset) if eval_dataset is not None else 0:,}")
print(
    "Approx example character length: "
    f"p50={percentile(example_char_lengths, 50):,}, "
    f"p90={percentile(example_char_lengths, 90):,}, "
    f"max={max(example_char_lengths):,}"
)
print(
    "Approx answer character length: "
    f"p50={percentile(answer_char_lengths, 50):,}, "
    f"p90={percentile(answer_char_lengths, 90):,}, "
    f"max={max(answer_char_lengths):,}"
)
print("\nRedacted example structure:\n")
print(make_redacted_example(example_shapes[0]))


## Load Tokenizer and Model

This section loads the selected base model with P100-compatible settings: fp16, no bf16, no FlashAttention assumptions, and optional 4-bit loading.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample_for_lengths = train_dataset.select(range(min(len(train_dataset), 1000)))
token_lengths = []
for prompt, answer in zip(sample_for_lengths["prompt"], sample_for_lengths["answer"]):
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"][:MAX_ANSWER_TOKENS]
    token_lengths.append(min(MAX_SEQ_LENGTH, len(prompt_ids) + len(answer_ids) + 1))

print(
    "Approx token length after truncation: "
    f"p50={percentile(token_lengths, 50):,}, "
    f"p90={percentile(token_lengths, 90):,}, "
    f"max={max(token_lengths) if token_lengths else 0:,}"
)

compute_dtype = torch.float16
quantization_config = None
if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

model_load_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto",
    "torch_dtype": compute_dtype,
}
if quantization_config is not None:
    model_load_kwargs["quantization_config"] = quantization_config

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_load_kwargs)

if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(model)

try:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
except TypeError:
    model.gradient_checkpointing_enable()

model.config.use_cache = False
print(f"Loaded {MODEL_NAME}")
print(f"4-bit loading: {LOAD_IN_4BIT}")
print("Training dtype: fp16")


## Configure LoRA

The adapter targets common attention projection modules used by the selected model presets.


In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Train

This uses a compact `Trainer` subclass that computes loss only on answer tokens. When the selected model supports limiting returned logits, the trainer requests only the final answer-region logits to reduce memory further.


In [ ]:
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments


def filtered_kwargs(cls, kwargs):
    params = inspect.signature(cls.__init__).parameters
    return {key: value for key, value in kwargs.items() if key in params}


def add_eval_strategy(kwargs, params, has_eval):
    if "eval_strategy" in params:
        kwargs["eval_strategy"] = "steps" if has_eval else "no"
    elif "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = "steps" if has_eval else "no"
    if has_eval:
        kwargs["eval_steps"] = EVAL_STEPS


def training_kwargs(arg_cls, has_eval):
    params = inspect.signature(arg_cls.__init__).parameters
    kwargs = {
        "output_dir": OUTPUT_DIR,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "warmup_ratio": WARMUP_RATIO,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "logging_steps": LOGGING_STEPS,
        "save_steps": SAVE_STEPS,
        "save_strategy": "steps",
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "prediction_loss_only": True,
        "group_by_length": True,
        "remove_unused_columns": False,
        "max_grad_norm": MAX_GRAD_NORM,
        "report_to": "none",
        "seed": SEED,
        "bf16": False,
        "fp16": True,
    }
    if LOAD_IN_4BIT and "optim" in params:
        kwargs["optim"] = "paged_adamw_8bit"
    add_eval_strategy(kwargs, params, has_eval)
    return filtered_kwargs(arg_cls, kwargs)


def tokenize_for_completion_loss(batch):
    output = {"input_ids": [], "attention_mask": [], "labels": []}
    eos_id = tokenizer.eos_token_id

    for prompt, answer in zip(batch["prompt"], batch["answer"]):
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"][:MAX_ANSWER_TOKENS]
        if eos_id is not None:
            answer_ids = answer_ids + [eos_id]

        if len(answer_ids) >= MAX_SEQ_LENGTH:
            answer_ids = answer_ids[: MAX_SEQ_LENGTH - 1]
            if eos_id is not None:
                answer_ids.append(eos_id)

        available_for_prompt = max(0, MAX_SEQ_LENGTH - len(answer_ids))
        prompt_ids = prompt_ids[-available_for_prompt:] if available_for_prompt > 0 else []

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids.copy()

        output["input_ids"].append(input_ids)
        output["attention_mask"].append([1] * len(input_ids))
        output["labels"].append(labels)

    return output


def completion_data_collator(features):
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    max_length = max(len(feature["input_ids"]) for feature in features)
    max_length = ((max_length + 7) // 8) * 8

    batch = {"input_ids": [], "attention_mask": [], "labels": []}
    for feature in features:
        pad_length = max_length - len(feature["input_ids"])
        batch["input_ids"].append(feature["input_ids"] + [pad_id] * pad_length)
        batch["attention_mask"].append(feature["attention_mask"] + [0] * pad_length)
        batch["labels"].append(feature["labels"] + [-100] * pad_length)

    return {key: torch.tensor(value, dtype=torch.long) for key, value in batch.items()}


def find_logits_keep_arg(peft_model):
    candidate = peft_model.get_base_model() if hasattr(peft_model, "get_base_model") else peft_model
    try:
        params = inspect.signature(candidate.forward).parameters
    except (TypeError, ValueError):
        return None
    if "num_logits_to_keep" in params:
        return "num_logits_to_keep"
    if "logits_to_keep" in params:
        return "logits_to_keep"
    return None


LOGITS_KEEP_ARG = find_logits_keep_arg(model)
LOGITS_TO_KEEP = min(MAX_SEQ_LENGTH, MAX_ANSWER_TOKENS + 2)
print(f"Forward logits limit argument: {LOGITS_KEEP_ARG or 'not supported'}")


class CompletionOnlyTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        forward_kwargs = {}
        if LOGITS_KEEP_ARG is not None:
            forward_kwargs[LOGITS_KEEP_ARG] = LOGITS_TO_KEEP

        outputs = model(**inputs, **forward_kwargs)
        logits = outputs.logits

        if logits.shape[1] == labels.shape[1]:
            shift_logits = logits[:, :-1, :]
            shift_labels = labels[:, 1:]
        else:
            shift_logits = logits[:, :-1, :]
            label_start = labels.shape[1] - shift_logits.shape[1]
            shift_labels = labels[:, label_start:]

        active_positions = shift_labels.ne(-100)
        if active_positions.any():
            active_logits = shift_logits[active_positions]
            active_labels = shift_labels[active_positions]
            loss = F.cross_entropy(active_logits, active_labels, reduction="mean")
        else:
            loss = shift_logits.sum() * 0.0

        return (loss, outputs) if return_outputs else loss


has_eval = eval_dataset is not None and len(eval_dataset) > 0

tokenized_train = train_dataset.map(
    tokenize_for_completion_loss,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_eval = None
if has_eval:
    tokenized_eval = eval_dataset.map(
        tokenize_for_completion_loss,
        batched=True,
        remove_columns=eval_dataset.column_names,
    )

training_args = TrainingArguments(**training_kwargs(TrainingArguments, has_eval))
trainer = CompletionOnlyTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=completion_data_collator,
)

gc.collect()
torch.cuda.empty_cache()

print("Starting LoRA training")
trainer.train()


## Save Adapter

The LoRA adapter, tokenizer, and a small run configuration file are saved to `/kaggle/working/` so they appear as Kaggle notebook output artifacts.


In [ ]:
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

run_config = {
    "model_preset": MODEL_PRESET,
    "model_name": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_answer_tokens": MAX_ANSWER_TOKENS,
    "max_context_messages": MAX_CONTEXT_MESSAGES,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
}
(output_path / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Saved adapter to: {OUTPUT_DIR}")
print("Output files:")
for path in sorted(output_path.iterdir()):
    print(path)


In [ ]:
del trainer
try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_load_kwargs)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()
model.config.use_cache = True
print("Reloaded base model with trained LoRA adapter.")


## Test Generation

This section uses synthetic contexts only. It does not print or save raw Telegram training examples.


In [ ]:
def normalize_role(role):
    role = str(role).strip().lower()
    if role in {"friend", "assistant", "model"}:
        return "Friend"
    return "User"


def format_inference_prompt(context_messages):
    lines = []
    for role, text in context_messages[-MAX_CONTEXT_MESSAGES:]:
        clean_text = clean_message_text(text)
        if clean_text:
            lines.append(f"{normalize_role(role)}: {clean_text}")
    return f"{SYSTEM_PROMPT}\n<|context|>\n" + "\n".join(lines) + "\n<|answer|>\n"


def clean_generated_reply(text):
    if tokenizer.eos_token:
        text = text.replace(tokenizer.eos_token, "")
    for stop_marker in ["\nUser:", "\nFriend:", "<|context|>", "<|answer|>", "<|system|>"]:
        if stop_marker in text:
            text = text.split(stop_marker, 1)[0]
    return text.strip()


def generate_friend_reply(
    context_messages,
    max_new_tokens=80,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.05,
):
    prompt = format_inference_prompt(context_messages)
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return clean_generated_reply(generated_text)


synthetic_contexts = [
    [("User", "Hey, are you free later today?")],
    [("User", "I just finished the thing we talked about."), ("Friend", "Nice"), ("User", "Want me to send it?")],
    [("User", "What do you think about getting coffee this weekend?")],
]

generation_records = []
for index, context in enumerate(synthetic_contexts, start=1):
    reply = generate_friend_reply(context)
    generation_records.append({"example": index, "context": context, "reply": reply})
    print(f"Example {index}: {reply[:240]}")

Path(GENERATION_EXAMPLES_PATH).write_text(
    json.dumps(generation_records, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Saved generation examples to: {GENERATION_EXAMPLES_PATH}")


## Interactive Chat

Kaggle notebooks are not always reliable for hosted Gradio interfaces, so this section provides both a plain Python chat helper and an optional Gradio UI.


In [ ]:
def chat_once(user_message, history=None):
    history = [] if history is None else list(history)
    context = []
    for previous_user_message, previous_friend_reply in history[-MAX_CONTEXT_MESSAGES:]:
        if previous_user_message:
            context.append(("User", previous_user_message))
        if previous_friend_reply:
            context.append(("Friend", previous_friend_reply))
    context.append(("User", user_message))
    reply = generate_friend_reply(context)
    history.append((user_message, reply))
    return reply, history


def run_text_chat():
    history = []
    print("Type 'exit' to stop.")
    while True:
        user_message = input("You: ").strip()
        if user_message.lower() in {"exit", "quit", "q"}:
            break
        reply, history = chat_once(user_message, history)
        print(f"Friend: {reply}")
    return history


history = []
reply, history = chat_once("Hi, how is your day going?", history)
print(reply)


In [ ]:
ENABLE_GRADIO = False

if ENABLE_GRADIO:
    try:
        import gradio as gr

        def gradio_reply(message, history):
            context = []
            for item in history or []:
                role = "Friend" if item.get("role") == "assistant" else "User"
                context.append((role, item.get("content", "")))
            context.append(("User", message))
            return generate_friend_reply(context)

        demo = gr.ChatInterface(
            fn=gradio_reply,
            type="messages",
            title="Friend LoRA Chat",
            description="Runs inside this Kaggle notebook session. Review outputs before sharing.",
        )
        demo.launch(share=False, debug=False)
    except Exception as exc:
        print(f"Gradio could not start: {type(exc).__name__}: {exc}")
        print("Use chat_once(...) or run_text_chat() instead.")
else:
    print("Set ENABLE_GRADIO = True and rerun this cell to try the Gradio chat UI.")
    print("Plain Python fallback is available through chat_once(...) and run_text_chat().")
